# 02 - Constructing the Winger Population

## Decoding Winger Playstyles

This notebook constructs the player population used for the winger playstyle analysis.

The source FBref dataset only categorises players into broad positional groups (`FW`, `MF`, `DF`, `GK`). This is insufficient for the project because the `FW` category contains both central forwards and wide attackers, while some wingers may also be categorised as midfielders.

Rather than identifying wingers from their statistical behaviour, which would introduce circularity into the later style analysis, an independent auxiliary historical FIFA/SoFIFA dataset is therefore used to identify wide players more precisely.

The objective of this notebook is therefore to:

1. load the audited FBref dataset,
2. identify the broad pool of potential attacking players,
3. enrich players with independent detailed position information,
4. define and document a winger inclusion rule,
5. apply playing-time requirements,
6. manually sanity-check the resulting population,
7. save the final winger dataset for later feature engineering and modelling.

No playstyle modelling is performed in this notebook.

## 1. Load the audited player dataset

The cleaned dataset produced in Notebook 01 is used as the starting point.

Each observation represents a **player × club × season** spell rather than necessarily a complete player-season.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("fbref_2017_2024_audited.csv")

print("Rows:", len(df))
print("Columns:", df.shape[1])

df

## 2. Build the broad attacking-player candidate pool

The source data only provides four broad positions.

Potential wingers may be listed as either:

- `FW` — forwards
- `MF` — midfielders

Both groups are retained at this stage.

No player is classified as a winger based on their statistical profile.

In [ ]:
candidate_pool = df[df["pos"].isin(["FW", "MF"])].copy()

print("Candidate observations:", len(candidate_pool))

display(
    candidate_pool["pos"]
    .value_counts()
    .rename("observations")
    .to_frame()
)

In [ ]:
candidate_pool[["player_id", "player", "season", "squad", "comp", "pos", "Minutes"]].head(20)

## 3. Prepare the auxiliary positional data

The FBref source only identifies players as forwards, midfielders, defenders or goalkeepers and therefore cannot distinguish wingers from central attackers.

Detailed positional information is taken from the historical EA Sports / SoFIFA player dataset.

For this project, a **winger** is defined as a player whose primary listed position OR club position for the corresponding FIFA edition is one of:

- `LW` — Left Winger
- `RW` — Right Winger
- `LM` — Left Midfielder
- `RM` — Right Midfielder

`LM` and `RM` are included because older FIFA editions and traditional formations frequently represent wide attacking players as wide midfielders rather than forwards.

Only positional metadata is taken from the FIFA dataset. FIFA ability ratings and attributes are not used in the football analysis.

In [ ]:
fifa = pd.read_csv("male_players.csv", low_memory=False)

fifa_positions = fifa[
    [
        "player_id",
        "short_name",
        "long_name",
        "fifa_version",
        "dob",
        "nationality_name",
        "club_name",
        "league_name",
        "player_positions",
        "club_position"
    ]
].copy()

fifa_positions["primary_position"] = (
    fifa_positions["player_positions"]
    .str.split(",")
    .str[0]
    .str.strip()
)

fifa_positions["birth_year"] = pd.to_datetime(
    fifa_positions["dob"]
).dt.year

In [ ]:
fifa_positions = fifa_positions[
    fifa_positions["fifa_version"].between(18, 24)
].copy()

In [ ]:
fifa_positions.head(5)

## 4. Align football seasons with FIFA editions

Each FBref domestic season is mapped to the FIFA edition released during that season.

For example, the 2017/18 season corresponds to FIFA 18 and the 2023/24 season corresponds to EA Sports FC 24.

In [ ]:
season_to_fifa = {
    "2017-2018": 18,
    "2018-2019": 19,
    "2019-2020": 20,
    "2020-2021": 21,
    "2021-2022": 22,
    "2022-2023": 23,
    "2023-2024": 24
}

candidate_pool["fifa_version"] = (
    candidate_pool["season"]
    .map(season_to_fifa)
)

In [ ]:
fifa_to_season = {
    version: season
    for season, version in season_to_fifa.items()
}

fifa_positions["season"] = (
    fifa_positions["fifa_version"]
    .map(fifa_to_season)
)

In [ ]:
WINGER_POSITIONS = ["LW", "RW", "LM", "RM"]

fifa_positions["is_winger"] = (
    fifa_positions["primary_position"]
    .isin(WINGER_POSITIONS)
)

## 5. Match player identities between FBref and FIFA Datasets

The two datasets do not share a common player identifier and use different naming conventions.

For example, FBref may record `Mohamed Salah`, while FIFA records the same player as `Mohamed Salah Ghaly`.

The final identity matcher resolves each distinct FBref footballer once. It restricts candidates by birth year and compares normalized short- and long-name aliases pooled across FIFA editions 18–24. The highest name score wins; similarity across the available club histories breaks ties.

Only a unique best footballer with a name score of 100 is accepted automatically; the remaining eligible identities require the saved manual review. Season-specific positions are then attached using the resolved FIFA identity and the season-to-edition mapping.

The earlier season-specific `find_fifa_match` function is exploratory; it is not the final matcher used to construct the population.

Match scores are retained so uncertain cases can be reviewed rather than silently accepted.

In [ ]:
!pip install rapidfuzz -q

In [ ]:
from rapidfuzz.fuzz import token_set_ratio, WRatio
import re
import unicodedata

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
candidate_pool["name_norm"] = (
    candidate_pool["player"]
    .apply(normalize_text)
)

candidate_pool["club_norm"] = (
    candidate_pool["squad"]
    .apply(normalize_text)
)

fifa_positions["short_name_norm"] = (
    fifa_positions["short_name"]
    .apply(normalize_text)
)

fifa_positions["long_name_norm"] = (
    fifa_positions["long_name"]
    .apply(normalize_text)
)

fifa_positions["club_norm"] = (
    fifa_positions["club_name"]
    .apply(normalize_text)
)

In [ ]:
candidate_pool[["name_norm","club_norm"]].head(5)

In [ ]:
WINGER_POSITIONS = {"LW", "RW", "LM", "RM"}

fifa_positions["is_winger"] = (
    fifa_positions["primary_position"].isin(WINGER_POSITIONS)
    | fifa_positions["club_position"].isin(WINGER_POSITIONS)
)

In [ ]:
fifa_positions[["long_name_norm","club_norm","primary_position","club_position","is_winger"]].sample(n=10)

In [ ]:
def find_fifa_match(row, fifa_df):

    candidates = fifa_df[
        (fifa_df["fifa_version"] == row["fifa_version"])
        & (fifa_df["birth_year"] == row["born"])
    ].copy()

    if candidates.empty:
        return pd.Series({
            "fifa_player_id": pd.NA,
            "fifa_name": pd.NA,
            "fifa_club": pd.NA,
            "primary_position": pd.NA,
            "club_position": pd.NA,
            "is_winger": pd.NA,
            "name_score": np.nan,
            "club_score": np.nan,
            "match_score": np.nan
        })

    candidates["name_score"] = candidates.apply(
        lambda x: max(
            token_set_ratio(
                row["name_norm"],
                x["short_name_norm"]
            ),
            token_set_ratio(
                row["name_norm"],
                x["long_name_norm"]
            )
        ),
        axis=1
    )

    candidates["club_score"] = candidates[
        "club_norm"
    ].apply(
        lambda x: WRatio(row["club_norm"], x)
    )

    # Player identity is driven primarily by name.
    # Club acts only as a supporting/tie-breaking signal.
    candidates["match_score"] = (
        0.9 * candidates["name_score"]
        + 0.1 * candidates["club_score"]
    )

    best = candidates.sort_values(
        "match_score",
        ascending=False
    ).iloc[0]

    return pd.Series({
        "fifa_player_id": best["player_id"],
        "fifa_name": best["long_name"],
        "fifa_club": best["club_name"],
        "primary_position": best["primary_position"],
        "club_position": best["club_position"],
        "is_winger": best["is_winger"],
        "name_score": best["name_score"],
        "club_score": best["club_score"],
        "match_score": best["match_score"]
    })

In [ ]:
fbref_identities = (
    candidate_pool
    .groupby(["player_id", "born"], as_index=False)
    .agg(
        player=("player", "first"),
        name_norm=("name_norm", "first"),
        fbref_clubs=(
            "club_norm",
            lambda x: list(pd.unique(x))
        )
    )
)

print("FBref observations:", len(candidate_pool))
print("Unique footballers to match:", len(fbref_identities))

In [ ]:
fifa_identities = (
    fifa_positions
    .groupby(["player_id", "birth_year"], as_index=False)
    .agg(
        fifa_name=("long_name", "last"),
        long_names=(
            "long_name_norm",
            lambda x: list(pd.unique(x))
        ),
        short_names=(
            "short_name_norm",
            lambda x: list(pd.unique(x))
        ),
        fifa_clubs=(
            "club_norm",
            lambda x: list(pd.unique(x))
        )
    )
)

In [ ]:
from rapidfuzz import process, fuzz

In [ ]:
name_choices_by_birth_year = {}

for birth_year, group in fifa_identities.groupby("birth_year"):

    choices = {}

    for row in group.itertuples():

        for i, name in enumerate(row.long_names):
            if name:
                choices[(row.player_id, "long", i)] = name

        for i, name in enumerate(row.short_names):
            if name:
                choices[(row.player_id, "short", i)] = name

    name_choices_by_birth_year[birth_year] = choices

In [ ]:
fifa_clubs_lookup = (
    fifa_identities
    .set_index("player_id")["fifa_clubs"]
    .to_dict()
)

fifa_name_lookup = (
    fifa_identities
    .set_index("player_id")["fifa_name"]
    .to_dict()
)

### Creating the identity matcher

In [ ]:
from rapidfuzz import process, fuzz

def match_fbref_identity(row):

    choices = name_choices_by_birth_year.get(row["born"])

    # No FIFA candidates with this birth year
    if not choices:
        return pd.Series({
            "fifa_player_id": pd.NA,
            "fifa_name": pd.NA,
            "name_score": np.nan,
            "club_score": np.nan,
            "name_tie_count": 0
        })

    # Find the strongest name candidates
    results = process.extract(
        row["name_norm"],
        choices,
        scorer=fuzz.token_set_ratio,
        limit=10
    )

    # A FIFA player may appear more than once because
    # we search both short_name and long_name.
    # Keep only that player's best name score.
    player_scores = {}

    for _, score, key in results:

        fifa_player_id = key[0]

        player_scores[fifa_player_id] = max(
            score,
            player_scores.get(fifa_player_id, 0)
        )

    # Find the highest name score achieved
    best_name_score = max(
        player_scores.values()
    )

    # Count how many DIFFERENT FIFA players
    # achieved that same best name score
    name_tie_count = sum(
        np.isclose(score, best_name_score)
        for score in player_scores.values()
    )

    ranked_candidates = []

    for fifa_player_id, name_score in player_scores.items():

        fifa_clubs = fifa_clubs_lookup.get(
            fifa_player_id, []
        )

        if row["fbref_clubs"] and fifa_clubs:

            club_score = max(
                fuzz.token_set_ratio(
                    fbref_club,
                    fifa_club
                )
                for fbref_club in row["fbref_clubs"]
                for fifa_club in fifa_clubs
            )

        else:
            club_score = 0

        ranked_candidates.append(
            (
                fifa_player_id,
                name_score,
                club_score
            )
        )

    # Highest name score wins.
    # Club similarity breaks ties.
    best = max(
        ranked_candidates,
        key=lambda x: (x[1], x[2])
    )

    fifa_player_id, name_score, club_score = best

    return pd.Series({
        "fifa_player_id": fifa_player_id,
        "fifa_name": fifa_name_lookup[fifa_player_id],
        "name_score": name_score,
        "club_score": club_score,
        "name_tie_count": name_tie_count
    })

### Run the identity matcher

Each unique FBref player is matched once to a stable FIFA player identity. Season-specific positional information will be attached only after identity matching is complete.

In [ ]:
match_columns = [
    "fifa_player_id",
    "fifa_name",
    "name_score",
    "club_score",
    "name_tie_count"
]

fbref_identity_base = fbref_identities.drop(
    columns=match_columns,
    errors="ignore"
).copy()

identity_matches = fbref_identity_base.apply(
    match_fbref_identity,
    axis=1
)

fbref_identities = pd.concat(
    [
        fbref_identity_base.reset_index(drop=True),
        identity_matches.reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
fbref_identities[
    [
        "player",
        "fifa_name",
        "name_score",
        "club_score",
        "name_tie_count"
    ]
].head()

## 6. Apply a minimum playing-time requirement

Player style should be inferred from a meaningful body of playing time rather than a small number of appearances.

A minimum threshold of **1,125 minutes** is applied to each player-club-season observation, equivalent to 12.5 full 90-minute matches.

Because the observational unit is player × club × season, the threshold is applied separately to each club spell.

In [ ]:
MIN_MINUTES = 1125

eligible_pool = candidate_pool[
    candidate_pool["Minutes"] >= MIN_MINUTES
].copy()

print("Candidate observations:", len(candidate_pool))
print("Eligible observations:", len(eligible_pool))
print(
    "Eligible unique players:",
    eligible_pool["player_id"].nunique()
)

### Restrict identity validation to eligible players

Only players with at least 1,125 minutes in one or more player-club-season observations can enter the final analysis.

Identity-match validation is therefore restricted to this population.

In [ ]:
eligible_player_ids = eligible_pool[
    "player_id"
].unique()

eligible_matches = fbref_identities[
    fbref_identities["player_id"].isin(
        eligible_player_ids
    )
].copy()

print(
    "Eligible player identities:",
    len(eligible_matches)
)

### Identity-match quality control

An identity is accepted automatically when:

- normalized token-based name similarity is 100, and
- only one FIFA player achieves that best score.

All other cases are flagged for review rather than being force-assigned.

Club history is retained as a supporting validation signal, but is not used as the primary identity criterion.

In [ ]:
eligible_matches["match_status"] = np.where(
    (eligible_matches["name_score"] == 100)
    & (eligible_matches["name_tie_count"] == 1),
    "automatic",
    "review"
)

In [ ]:
match_summary = (
    eligible_matches["match_status"]
    .value_counts()
    .rename_axis("Match Status")
    .reset_index(name="Players")
)

match_summary

In [ ]:
review_matches = eligible_matches[
    eligible_matches["match_status"] == "review"
].copy()

print("Players requiring review:",len(review_matches))

In [ ]:
review_export = review_matches[
    [
        "player_id",
        "player",
        "born",
        "fbref_clubs",
        "fifa_player_id",
        "fifa_name",
        "name_score",
        "club_score",
        "name_tie_count"
    ]
].copy()

review_export["decision"] = ""
review_export["resolved_fifa_player_id"] = ""
review_export["notes"] = ""

# Templates are separate from authoritative completed decisions.
# Exclusive creation also protects any work already entered into a template.
template_path = Path("identity_matches_for_review_template.csv")
try:
    with template_path.open("x", encoding="utf-8", newline="") as handle:
        review_export.to_csv(handle, index=False)
except FileExistsError:
    print(f"Preserved existing template: {template_path}")

Non-automatic identity matches were reviewed outside the notebook and stored in a small audit file.

Confirmed matches were retained, incorrect matches were corrected where possible, and unresolved FIFA identities were left unmatched.

In [ ]:
reviewed_path = Path("identity_matches_for_review.csv")
if not reviewed_path.is_file():
    raise FileNotFoundError(
        f"Missing authoritative reviewed input: {reviewed_path.resolve()}. "
        "Supply the completed review file from data/raw in the working "
        "directory. A blank *_template.csv is not a substitute; "
        "reviewed decisions will not be regenerated automatically."
    )

manual_review = pd.read_csv(reviewed_path)

manual_review["decision"].value_counts(dropna=False)

In [ ]:
assert manual_review["decision"].notna().all()

In [ ]:
automatic_matches = eligible_matches[
    eligible_matches["match_status"] == "automatic"
][
    ["player_id", "fifa_player_id"]
].copy()

automatic_matches["match_method"] = "automatic"

In [ ]:
manual_accepts = manual_review[
    manual_review["decision"] == "accept"
][
    ["player_id", "fifa_player_id"]
].copy()

manual_accepts["match_method"] = "manual"

In [ ]:
identity_map = pd.concat(
    [
        automatic_matches,
        manual_accepts
    ],
    ignore_index=True
)

In [ ]:
identity_map["fifa_player_id"] = pd.to_numeric(
    identity_map["fifa_player_id"],
    errors="coerce"
).astype("Int64")

In [ ]:
print("Resolved identities:", len(identity_map))

assert not identity_map["player_id"].duplicated().any()

In [ ]:
eligible_pool = eligible_pool.merge(
    identity_map,
    on="player_id",
    how="left",
    validate="many_to_one"
)

In [ ]:
print("Eligible observations:", len(eligible_pool))

print(
    "Without FIFA identity:",
    eligible_pool["fifa_player_id"].isna().sum()
)

In [ ]:
season_positions = fifa_positions[
    [
        "player_id",
        "fifa_version",
        "primary_position",
        "club_position",
        "is_winger"
    ]
].copy()

In [ ]:
season_positions = season_positions.rename(
    columns={
        "player_id": "fifa_player_id"
    }
)

season_positions["fifa_player_id"] = (
    pd.to_numeric(
        season_positions["fifa_player_id"],
        errors="coerce"
    )
    .astype("Int64")
)

In [ ]:
eligible_pool = eligible_pool.merge(
    season_positions,
    on=["fifa_player_id", "fifa_version"],
    how="left",
    validate="many_to_one"
)

## 7. Resolve missing positional classifications

FIFA is used as an auxiliary positional source rather than as an exhaustive record of all professional players.

A small number of eligible FBref observations could not receive a reliable season-specific positional classification from FIFA because the player was either absent from the corresponding FIFA edition or could not be matched confidently.

These remaining observations were manually verified using **historical season-specific Transfermarkt positional data**.

Classification was performed at the **player × club × season** level rather than assigning one permanent position to a player.

The same winger definition was retained:

- `LW`
- `RW`
- `LM`
- `RM`

Players whose historical role fell outside these positions were classified as non-wingers.

The manual decisions are stored separately in `manual_position_overrides.csv` to preserve an auditable record of the classifications.

In [ ]:
lamine_mask = (
    (eligible_pool["player_id"] == "Lamine Yamal_2007")
    & (eligible_pool["season"] == "2023-2024")
)

eligible_pool.loc[
    lamine_mask,
    "primary_position"
] = "RW"

eligible_pool.loc[
    lamine_mask,
    "club_position"
] = "RW"

eligible_pool.loc[
    lamine_mask,
    "is_winger"
] = True

In [ ]:
eligible_pool["position_source"] = np.where(
    eligible_pool["fifa_player_id"].notna(),
    "FIFA",
    pd.NA
)

eligible_pool.loc[
    lamine_mask,
    "position_source"
] = "Manual override"

In [ ]:
unclassified = eligible_pool[
    eligible_pool["is_winger"].isna()
][
    [
        "player_id",
        "player",
        "season",
        "squad",
        "Minutes"
    ]
].drop_duplicates()

unclassified

In [ ]:
unclassified_review = (
    eligible_pool[
        eligible_pool["is_winger"].isna()
    ][
        [
            "player_id",
            "player",
            "season",
            "squad",
            "Minutes"
        ]
    ]
    .drop_duplicates()
    .copy()
)

unclassified_review["manual_position"] = ""
unclassified_review["is_winger_manual"] = ""
unclassified_review["source"] = ""
unclassified_review["notes"] = ""

# Templates are separate from authoritative completed decisions.
# Exclusive creation also protects any work already entered into a template.
template_path = Path("manual_position_overrides_template.csv")
try:
    with template_path.open("x", encoding="utf-8", newline="") as handle:
        unclassified_review.to_csv(handle, index=False)
except FileExistsError:
    print(f"Preserved existing template: {template_path}")

In [ ]:
reviewed_path = Path("manual_position_overrides.csv")
if not reviewed_path.is_file():
    raise FileNotFoundError(
        f"Missing authoritative reviewed input: {reviewed_path.resolve()}. "
        "Supply the completed review file from data/raw in the working "
        "directory. A blank *_template.csv is not a substitute; "
        "reviewed decisions will not be regenerated automatically."
    )

manual_positions = pd.read_csv(reviewed_path)

manual_positions["is_winger_manual"] = (
    manual_positions["is_winger_manual"]
    .astype(str)
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)

In [ ]:
eligible_pool = eligible_pool.merge(
    manual_positions[
        [
            "player_id",
            "season",
            "squad",
            "manual_position",
            "is_winger_manual",
            "source",
            "notes"
        ]
    ],
    on=["player_id", "season", "squad"],
    how="left",
    validate="one_to_one"
)

In [ ]:
manual_mask = (
    eligible_pool["is_winger"].isna()
    & eligible_pool["is_winger_manual"].notna()
)

eligible_pool.loc[
    manual_mask,
    "is_winger"
] = eligible_pool.loc[
    manual_mask,
    "is_winger_manual"
]

eligible_pool.loc[
    manual_mask,
    "primary_position"
] = eligible_pool.loc[
    manual_mask,
    "manual_position"
]

In [ ]:
# Preserve the source labels established in cell 57, including
# Lamine Yamal's documented manual RW override. No historical URL is inferred.
eligible_pool.loc[
    manual_mask,
    "position_source"
] = "Manual verification"

In [ ]:
eligible_pool["is_winger"].isna().sum()

In [ ]:
unclassified_count = (
    eligible_pool[
        "is_winger"
    ]
    .isna()
    .sum()
)

print(
    "Unclassified observations:",
    unclassified_count
)

In [ ]:
assert unclassified_count == 0

## 8. Construct the final winger population

After combining FIFA positional metadata with manually verified historical positions, every eligible player-club-season observation has a winger/non-winger classification.

The final analysis population retains observations classified as wide players and satisfying the minimum requirement of 1,125 minutes.

In [ ]:
winger_pool = eligible_pool[
    eligible_pool[
        "is_winger"
    ].eq(True)
].copy()

In [ ]:
print(
    "Winger observations:",
    len(winger_pool)
)

print(
    "Unique players:",
    winger_pool[
        "player_id"
    ].nunique()
)

print(
    "Seasons covered:",
    winger_pool[
        "season"
    ].nunique()
)

In [ ]:
winger_pool[
    "season"
].value_counts().sort_index()

In [ ]:
winger_pool[
    "primary_position"
].value_counts(
    dropna=False
)

In [ ]:
winger_pool[
    "position_source"
].value_counts()

Sanity Check and Preview sample

In [ ]:
winger_pool[
    [
        "player",
        "season",
        "squad",
        "Minutes",
        "primary_position",
        "club_position",
        "position_source"
    ]
].sample(
    min(
        15,
        len(winger_pool)
    ),
    random_state=42
)

In [ ]:
assert (
    winger_pool["Minutes"]
    >= MIN_MINUTES
).all()

assert (
    winger_pool[
        "is_winger"
    ]
    .eq(True)
    .all()
)

assert (
    winger_pool[
        "season"
    ]
    .isin(
        season_to_fifa.keys()
    )
    .all()
)

In [ ]:
assert (
    winger_pool[
        [
            "player_id",
            "season",
            "squad",
            "comp"
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

## 9. Save the analysis population

The resulting dataset contains the player-club-season observations used in the subsequent playstyle analysis.

This notebook only defines **who belongs in the winger population**.

No playstyle features, dimensionality reduction, similarity modelling or clustering are performed here. Those steps are handled separately in the following notebooks.

In [ ]:
winger_pool.to_csv(
    "fbref_2017_2024_wingers.csv",
    index=False
)